# Table of Contents
- [0. The shopping list dataset from the literature](#The-shopping-list-dataset-from-the-literature)
- [1. Generating frequent patterns with the apriori algorithm](#1.-Generating-frequent-patterns)

In [1]:
import os
import pandas as pd

Frequent pattern analysis is beyond the scope of the `scikit-learn` library. In this notebook we will resort to `mlxtend` ([machine learning extension](http://rasbt.github.io/mlxtend/)), one of the third party libraries that implement the most popular frequent pattern mining algorithms.

In [2]:
from mlxtend.frequent_patterns import apriori

# The shopping list dataset from the literature

Load the shopping list dataset: it is the dataset from the literature with 9 transactions and 5 items

In [3]:
data = pd.read_csv('dataset/ShoppingList.csv', index_col=0)
data

,I1,I2,I3,I4,I5
TID,,,,,
T100,True,True,False,False,True
T200,False,True,False,True,False
T300,False,True,True,False,False
T400,True,True,False,True,False
T500,True,False,True,False,False
T600,False,True,True,False,False
T700,True,False,True,False,False
T800,True,True,True,False,True
T900,True,True,True,False,False


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9 entries, T100 to T900
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   I1      9 non-null      bool 
 1   I2      9 non-null      bool 
 2   I3      9 non-null      bool 
 3   I4      9 non-null      bool 
 4   I5      9 non-null      bool 
dtypes: bool(5)
memory usage: 117.0+ bytes


In [5]:
data.describe()

,I1,I2,I3,I4,I5
count,9,9,9,9,9
unique,2,2,2,2,2
top,True,True,True,False,False
freq,6,7,6,7,7


# 1. Generating frequent patterns

### The apriori algorithm

Apriori is a popular algorithm for extracting frequent itemsets with applications in association rule learning. The apriori algorithm has been designed to operate on databases containing transactions, such as purchases by customers of a store. 

An itemset is considered as "frequent" if it meets a user-specified support threshold. For instance, if the support threshold is set to 0.5 (50%), a frequent itemset is defined as a set of items that occur together in at least 50% of all transactions in the database.

Thus, to find all the frequent itemsets, we impose a predefined *minimum support* (min sup.).


**Encoded format**

The allowed values for a DataFrame provided as input at mlxtend's frequent pattern mining algorithms are either 0/1 or True/False (i.e., boolean vector). 

This encoding complies with the scenario of *market basket analysis*, in which we think of the universe as the set of items available at the store, and each item as a Boolean variable representing the presence or absence of that item. Each basket can be represented by a Boolean vector of values assigned to these variables.


In [6]:
from mlxtend.frequent_patterns import apriori

In [7]:
apriori?
# - requires setting of min_support
# - requires one-hot encoded dataframe

## Step 0: 
we choose a value for minsup 
## Step 1:
we apply the apriori algorithm

In [15]:
# get all itemsets with maximum length=1 AND min_support >= MinSup
MinSup = 0.2
freq_itemset_max1 = apriori(data, 
                       min_support = MinSup, 
                       use_colnames = True,
                       max_len=1,
                       verbose = True)
print("For Minsup =", MinSup, "and max_len=1, the a-priori result is:")
display(freq_itemset_max1)


For Minsup = 0.2 and max_len=1, the a-priori result is:


,support,itemsets
0,0.666667,(I1)
1,0.777778,(I2)
2,0.666667,(I3)
3,0.222222,(I4)
4,0.222222,(I5)


In [17]:
# get all itemsetswith min_support >=0.2

freq_itemset = apriori(data, 
                       min_support = 0.2, 
                       use_colnames = True,
                       verbose = True)

Processing 4 combinations | Sampling itemset size 4


In [18]:
freq_itemset

,support,itemsets
0,0.666667,(I1)
1,0.777778,(I2)
2,0.666667,(I3)
3,0.222222,(I4)
4,0.222222,(I5)
5,0.444444,"(I2, I1)"
6,0.444444,"(I3, I1)"
7,0.222222,"(I5, I1)"
8,0.444444,"(I3, I2)"
9,0.222222,"(I2, I4)"


In [65]:
sorted(freq_itemset.support.unique())   #get unique values of support (also, we order them)

[np.float64(0.2222222222222222),
 np.float64(0.4444444444444444),
 np.float64(0.6666666666666666),
 np.float64(0.7777777777777778)]

In [21]:
[(f'{x} / 9 = {x / 9}') for x in range(2, 10)]       

['2 / 9 = 0.2222222222222222',
 '3 / 9 = 0.3333333333333333',
 '4 / 9 = 0.4444444444444444',
 '5 / 9 = 0.5555555555555556',
 '6 / 9 = 0.6666666666666666',
 '7 / 9 = 0.7777777777777778',
 '8 / 9 = 0.8888888888888888',
 '9 / 9 = 1.0']

The type of the itemset value is `frozenset`.

In [67]:
freq_itemset.itemsets.values[0]

frozenset({'I1'})

Differently from the classical python `set`, the `frozenset` type is immutable and hashable — its contents cannot be altered after it is created; it can therefore be used as a dictionary key or as an element of another set. 
<br>*Note: sets are unchangeable: once a set is created, you cannot change the elements. BUT: you can remove or add new elements*

We can leverage the power of pandas to conveniently analyse/filter the results. For instance, we can create the DataFrame of frequent itemsets via apriori and then add a new column that stores the length of each itemset:

In [22]:
freq_itemset['length'] = freq_itemset['itemsets'].apply(lambda x: len(x))
freq_itemset

,support,itemsets,length
0,0.666667,(I1),1
1,0.777778,(I2),1
2,0.666667,(I3),1
3,0.222222,(I4),1
4,0.222222,(I5),1
5,0.444444,"(I2, I1)",2
6,0.444444,"(I3, I1)",2
7,0.222222,"(I5, I1)",2
8,0.444444,"(I3, I2)",2
9,0.222222,"(I2, I4)",2


Filter the results based on some desired criteria (e.g., selects only the *k*-itemset with *k*>2 and a minimum support)

In [28]:
freq_itemset[(freq_itemset['length'] > 2) & (freq_itemset['support'] >= 0.22)]

,support,itemsets,length
11,0.222222,"(I3, I2, I1)",3
12,0.222222,"(I5, I2, I1)",3


In [29]:
freq_itemset[freq_itemset['itemsets'].apply(lambda x: 'I1' in x)]

,support,itemsets,length
0,0.666667,(I1),1
5,0.444444,"(I2, I1)",2
6,0.444444,"(I3, I1)",2
7,0.222222,"(I5, I1)",2
11,0.222222,"(I3, I2, I1)",3
12,0.222222,"(I5, I2, I1)",3


Note: as we are dealing with frozensets, the order does not matter.

In [30]:
freq_itemset[freq_itemset['itemsets'] == {"I1", "I2"}]

,support,itemsets,length
5,0.444444,"(I2, I1)",2


In [31]:
freq_itemset[freq_itemset['itemsets'] == {"I2", "I1"}]


,support,itemsets,length
5,0.444444,"(I2, I1)",2
